# धडा ०५ - एजेंटिक RAG


## सेटअप

हा नोटबुक मायक्रोसॉफ्ट एजंट फ्रेमवर्क वापरून एजेंटिक RAG (रिट्रीव्हल-अगमेंटेड जनरेशन) पॅटर्न दाखवतो.

**पूर्वअटी:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — तुमचा Azure AI सर्च सेवा अंतिम बिंदू
- `AZURE_SEARCH_API_KEY` — तुमची Azure AI सर्च API की
- वातावरणातील चलांद्वारे कॉन्फिगर केलेले Azure OpenAI डिप्लॉयमेंट
- पुष्टी केलेले Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## एजेन्टिक RAG म्हणजे काय?

परंपरागत RAG एक निश्चित पाइपलाइनचे पालन करते: दस्तऐवज प्राप्त करा, नंतर प्रतिसाद तयार करा. **एजेन्टिक RAG** अधिक पुढे जाते आणि एजंटला माहिती कधी आणि कशी प्राप्त करायची हे ठरवण्याची स्वायत्तता देते.

एजेन्टिक RAG सह, एजंट करू शकतो:
- प्रश्नाला उत्तर देण्यापूर्वी प्राप्ती आवश्यक आहे का हे **निर्णय** करणे
- कोणत्या डेटा स्रोत किंवा साधनाला क्वेरी करायचे हे **निवडणे**
- प्राप्त झालेल्या निकालांचे **मूल्यमापन** करणे आणि जर पहिला प्रयत्न अपूर्ण असेल तर पुढील प्राप्ती करणे
- अनेक प्राप्ती टप्प्यांमधून माहिती एकत्र करून सुसंगत उत्तर तयार करणे

हे एजंटला स्थिर प्राप्ती-नंतर-निर्माण पाइपलाइनच्या तुलनेत अधिक लवचिक आणि अचूक बनवते.


## शोध साधन तयार करणे

Agentic RAG मध्ये, बाह्य डेटा स्रोतांना **साधने** म्हणून रॅप केले जाते जे एजंट आवश्यकतेनुसार कॉल करू शकतो. यामुळे एजंटला पुनर्प्राप्ती हा दुसरा क्रिया म्हणून मानता येतो, एका अनिवार्य टप्प्याऐवजी.

खाली आम्ही प्रवास ज्ञानभांडार परिभाषित करतो आणि ते एजंट शोधासाठी वापरू शकणाऱ्या साधनाच्या रूपात उघड करतो.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG एजंट तयार करणे

आता आपण एक एजंट तयार करतो ज्याला **उत्तर देण्याआधी नेहमी माहिती शोधावी** असे सूचित केले आहे. एजंट त्याच्या स्वतःच्या प्रशिक्षण डेटावर अवलंबून न राहता ज्ञानाधारात आधारित उत्तरे देण्यासाठी `search_travel_knowledge` साधनाचा वापर करतो.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## पुनरावर्ती पुन्हा मिळवणे — मेकऱ्याचे-चेकऱ्याचे पॅटर्न

Agentic RAG चे एक मुख्य फायदे म्हणजे **पुनरावर्ती पुनरावृत्ती**. एजंट आपल्या प्रारंभिक शोधांची पडताळणी, सुधारणा किंवा विस्तार करण्यासाठी अनेकदा शोध घेऊ शकतो — जसे की "मेकऱ्याचे-चेकऱ्याचे" कार्यप्रवाह:

1. **मेकऱ्याचा टप्पा**: एजंट प्रारंभिक माहिती मिळवतो आणि उत्तराचा मसुदा तयार करतो.
2. **चेकऱ्याचा टप्पा**: एजंट तपशीलांची पडताळणी करण्यासाठी किंवा रिकाम्या जागा भरण्यासाठी अतिरिक्त शोध करतो.

खाली, एजंटला एक प्रश्न विचारण्यात आला आहे ज्यासाठी अनेक ठिकाणांची तुलना करणे आवश्यक आहे, ज्यामुळे तो अनेक वेळा शोध घेण्यास प्रवृत्त होतो.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## सारांश

या धड्यात तुम्ही Microsoft Agent Framework वापरून **Agentic RAG** सिस्टम कसे तयार करायचे हे शिकलात:

- **Agentic RAG** एजंट्सना माहिती पुनर्प्राप्त करण्याचा निर्णय स्वायत्तपणे घेण्याची परवानगी देते, ज्यामुळे पुनर्प्राप्ती ठराविक नसून डायनॅमिक होते.
- **टूल्स डेटा स्रोत म्हणून**: बाह्य ज्ञान तळं (जसे की Azure AI Search) टूल्स म्हणून गुंडाळले जातात ज्यांना एजंट कॉल करू शकतो.
- **इटरेटिव्ह पुनर्प्राप्ती**: मेकर-चेककर पॅटर्न एजंटला अनेक पुनर्प्राप्ती फेऱ्या करण्याची परवानगी देते — शोधणे, पडताळणी करणे, आणि अंतिम उत्तर तयार करण्यापूर्वी सुधारणा करणे.

उत्पादनात, तुम्ही इन-मेमरी `TRAVEL_KNOWLEDGE_BASE` ची जागा वास्तविक Azure AI Search इंडेक्सने बदलाल जे मोठ्या प्रमाणावर प्रवास दस्तऐवज पुनर्प्राप्ती हाताळेल.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**सूचना**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून अनुवादित केला आहे. आम्ही अचूकतेसाठी प्रयत्नशील असताना, कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये चुका किंवा अचूकतेत गडबड असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानले पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतर शिफारस केली जाते. या भाषांतराचा वापर केल्यामुळे उद्भवणाऱ्या गैरसमजुती किंवा चुकीच्या व्याख्यांसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
